# Chapter 3: Working with Generative AI in Finance
**Financial AI in Practice** | Manning Publications

Chapter 3 follows the AlphaStream team as it builds the Alpha Digest with generative AI, one engineering decision at a
time. This notebook collects the chapter's five listings in order: a quick head-to-head "litmus test" of candidate
models on a real community post (3.1); a conceptual RAG loop that retrieves posts from a placeholder vector store and
grounds the prompt in them (3.2); a parameter-efficient fine-tuning (LoRA) setup with Hugging Face `transformers`,
`peft`, and `datasets` that teaches an open-weight model the AlphaStream voice (3.3); and two illustrative agent
snippets that contrast hand-written function calling (3.4) with the Model Context Protocol (3.5). The code mirrors the
printed listings and is conceptual, so read the note above each listing before running it: 3.1 and 3.2 call the
OpenAI API; 3.3 needs a CUDA GPU, a gated ~16 GB model download, and your own training file; and 3.4 and 3.5 are
pseudocode that the book says will not run as-is.

## Setup (not a book listing)
Listings 3.1 and 3.2 create `openai.OpenAI()`, which reads `OPENAI_API_KEY` from the environment. If you keep the key in a
`.env` file as described in appendix A.4, run the next cell first: `load_dotenv()` searches this folder and its
parents, so a `.env` file in the repository root is found. Running the listings sends real (billed) requests to the
OpenAI API.

In [ ]:
# Setup helper (not part of the printed listings): load OPENAI_API_KEY from a .env file
# in this folder or any parent folder (see appendix A.4).
import os
from dotenv import load_dotenv

load_dotenv()
print("OPENAI_API_KEY is set." if os.getenv("OPENAI_API_KEY")
      else "OPENAI_API_KEY is not set: openai.OpenAI() in listings 3.1 and 3.2 will raise an error.")

### Listing 3.1 — A simple, practical litmus test for model evaluation
*Section 3.3.1, Step 1 Choosing the foundation: The model selection dilemma (pp. 39–40)*

Sends the same anonymized community post to each candidate model and prints the one-sentence summaries side by side,
for a quick, qualitative comparison that goes beyond public benchmarks. As the comment in the listing says, update
`models_to_test` with the models available when you run it.

In [ ]:
import openai
from typing import List
import os

# This snippet demonstrates a practical test to compare different models
# on a core task, going beyond generic benchmarks.
client = openai.OpenAI() # Assumes OPENAI_API_KEY is set in the environment

def run_model_test(model_name: str, post: str) -> str:
    """Sends a single, domain-specific post to a model for a quality check."""
    prompt = f"""
    You are a financial analyst. Summarize the key investment thesis from this post
    in one clear sentence.

    Post: "{post}"

    Summary:
    """
    try:
        response = client.responses.create(
            model=model_name,
            input=prompt,
            temperature=0.1,
            max_output_tokens=100
        )
        return response.output_text.strip()
    except openai.APIError as e:
        return f"API error with model {model_name}: {e}"

# --- Example Test Case ---
test_post = "I'm still bullish on $TECHCO. Their recent acquisition in the AI space is a game-changer. PT $150."

# A list of candidate models to compare. You can update this list with
# the latest models available at the time of running.
models_to_test = ["gpt-4o", "gpt-4o-mini"]

print(f"--- Running Comparative Test for Post ---\n'{test_post}'\n")

for model in models_to_test:
    print(f"--- Testing Model: {model} ---")
    summary = run_model_test(model, test_post)
    print(f"Response: {summary}\n")

# In a real scenario, you would also add open-source models to this list
# and run them using a library like Hugging Face's transformers.

### Listing 3.2 — Conceptual snippet for a RAG-powered response
*Section 3.3.2, Step 2 Grounding the model in reality: Mastering RAG (pp. 42–44)*

Retrieve, augment, generate. `VectorStore` is a placeholder that returns two hard-coded posts; in a real application
it would query a vector database such as Pinecone or Chroma. The generation step calls `gpt-4o` through the Responses
API.

In [ ]:
import openai
from typing import List, Dict, Optional

class VectorStore:
    @staticmethod
    def search(user_id: str, k: int) -> List[Dict]:
        """
        Simulates a personalized search against the vector database for a specific user.
        """
        print(f"[VectorStore] Searching for posts relevant to user '{user_id}'...")
        # For this example, we return a hardcoded, illustrative result.
        return [
            {"content": "I'm still bullish on $TECHCO. Their recent acquisition is a game-changer. PT $150.", "author": "investor_A"},
            {"content": "Warning on $RETAILCORP. Supply chain issues are worsening.", "author": "investor_B"},
        ]


# Initialize the OpenAI client
client = openai.OpenAI()


def get_digest_with_rag(user_id: str) -> Optional[str]:
    """
    Generates a personalized digest using a robust RAG workflow.
    """
    print(f"Generating digest for user: {user_id}")

    # 1. Retrieval
    retrieved_posts = VectorStore.search(user_id=user_id, k=5)

    # Robust check for empty or null search results
    if not retrieved_posts:
        return "No new relevant posts found for your digest today."

    # 2. Augmentation
    context_str = "\n".join(
        [f'- Post from {post["author"]}: {post["content"]}' for post in retrieved_posts]
    )

    # A more specific prompt for higher quality output
    prompt = f"""
    You are a financial analyst. Based *only* on the following RECENT posts from investors,
    provide a concise 'Alpha Digest' that includes:
    - Key investment themes or theses.
    - Any notable price targets or significant sentiment changes.
    - A brief summary of the overall outlook implied by these posts.

    Do not use any prior knowledge beyond what is provided below.

    Relevant Posts:
    {context_str}

    Your Digest:
    """
    # 3. Generation
    try:
        # The Responses API is OpenAI's current primary interface.
        response = client.responses.create(
            model="gpt-4o",  # A current, widely available model
            input=prompt,
            max_output_tokens=300,
            temperature=0.2,
        )
        return response.output_text.strip()
    except openai.APIError as e:
        # Handle API-specific issues (network, rate limits, server errors, etc.).
        print(f"OpenAI API Error: {e}")
        return "The analysis service is currently unavailable due to a provider issue."


# --- Example Usage ---
digest = get_digest_with_rag(user_id="user-123")
print("\n--- Your Alpha Digest ---")
print(digest)

### Listing 3.3 — Conceptual snippet for fine-tuning with PEFT (LoRA) [5]
*Section 3.3.3, Step 3 Teaching our style: The art of fine-tuning (pp. 46–47)*

Prepares LoRA fine-tuning of an open-weight model on the team's curated digests. **Do not run this cell casually:**

- **Packages:** `torch`, `transformers==5.13.0`, `peft==0.19.1`, `datasets==5.0.0` (appendix A). `peft` also installs
  `accelerate`, which `device_map="auto"` and `Trainer` rely on.
- **Model access and download:** `meta-llama/Meta-Llama-3-8B` is a gated Hugging Face repository. Request access on
  the model page (Meta approves requests), then authenticate with `hf auth login` or the `HF_TOKEN` environment
  variable. The first run downloads about 16 GB of weights (four safetensors shards) into your Hugging Face cache. The
  comment in the listing invites you to substitute a newer open-weight model; the PEFT recipe is the same.
- **Hardware:** an NVIDIA GPU with CUDA is required. The 8B model alone needs about 16 GB of GPU memory in float16,
  and `fp16=True` mixed-precision training adds activations on top, so plan on at least 24 GB of GPU memory (more if
  your examples approach `max_length=1024`). It is not practical on a CPU or Apple-silicon laptop.
- **Data:** `alpha_stream_digests.jsonl` is not provided. The book's margin note: *"Your own curated digests, saved as
  JSONL (one "text" example per line); there is no public dataset."* Each line is a JSON object with a `text` field,
  for example `{"text": "Posts: ... Digest: ..."}`. Put the file next to this notebook.
- The printed listing stops after constructing the `Trainer`. Call `trainer.train()` yourself to start training;
  results go to `./alpha-digest-finetune-results`.

In [ ]:
import torch
from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

# 1. Load the Base Model and Tokenizer with Optimizations
# Use the newest Llama (or comparable open-weight) release available when you run this—the PEFT recipe is identical.
model_name = "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set a padding token if one doesn't exist. This is a common requirement for Llama models.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model with memory-saving optimizations
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16, # Use 16-bit precision to save memory
    device_map="auto"          # Automatically distribute the model across available GPUs
)

# 2. Configure a More Comprehensive LoRA Setup
# Target more layers to allow for more nuanced adaptation.
lora_config = LoraConfig(
    r=16, # Increased rank for more expressive power
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Target all attention projections
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters() # This will show it's still a tiny fraction of the total

# 3. Load the High-Quality, Domain-Specific Dataset
finetuning_dataset = load_dataset(
    "json", data_files="alpha_stream_digests.jsonl", split="train")

def tokenize_examples(examples):
    return tokenizer(examples["text"], truncation=True, max_length=1024)

tokenized_dataset = finetuning_dataset.map(
    tokenize_examples, batched=True, remove_columns=finetuning_dataset.column_names
)

# 4. Set Up the Trainer with Production-Ready Arguments
# A data collator handles batching and padding dynamically.
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./alpha-digest-finetune-results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # Simulate a larger batch size for stability
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,                     # Enable mixed-precision training
    logging_steps=10,
    warmup_steps=100,              # Learning rate warmup for smoother training
)

trainer = Trainer(
    model=peft_model,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    args=training_args,
)

### Listing 3.4 — The "Old Way": Building an agent with manual function calling
*Section 3.3.4, Step 4 The next frontier: From AI agents to agentic AI (p. 49)*

Shows the boilerplate of wiring tools by hand for traditional function calling. The book's margin note:
*"Illustrative pseudocode—will not run as-is; tool schemas and the agent body are omitted on purpose."* The `...`
placeholders inside the tool definitions are not valid Python, so this cell raises a `SyntaxError` by design. It
carries the standard `raises-exception` cell tag so that automated runs treat the error as expected.

In [ ]:
import openai
import json

# This snippet illustrates the manual, brittle nature of traditional function calling.
# A developer must implement and describe each tool individually.

# --- 1. Manually Implement Each Tool ---
def get_stock_chart(ticker: str):
    # ... code to call an internal charting service ...
    return {"chart_url": f"https://alphastream.com/charts/{ticker}.png"}

def search_community_posts(topic: str):
    # ... code to query the community database ...
    return {"posts": ["Post 1 about topic...", "Post 2 about topic..."]}

# --- 2. Manually Define and Handle Each Tool ---
tools = [ # Tool definitions for the LLM
    {"type": "function", "function": {"name": "get_stock_chart", ...}},
    {"type": "function", "function": {"name": "search_community_posts", ...}}
]

def run_manual_agent(user_prompt: str):
    # ... complex logic to call the LLM, check if it wants to use a tool,
    # execute the right function based on its name, handle arguments,
    # and potentially loop back to the LLM with the result ...
    print("Executing a complex, multi-step manual agent workflow...")
    # This part is deliberately omitted for brevity, as it is complex and brittle.
    return "Agent finished task (with a lot of custom code)."

print(run_manual_agent("Show me the chart for $TECHCO and recent posts about it."))

### Listing 3.5 — The "new way": a conceptual agent interacting via MCP
*Section 3.3.4, Step 4 The next frontier: From AI agents to agentic AI (pp. 50–51)*

The same agent expressed with the Model Context Protocol: the client discovers the tools exposed by MCP servers, and
the LLM decides which to call. As the book explains on p. 51, `mcp_client` is a placeholder for whichever MCP runtime
you adopt, not an installable package, so running this cell stops at the import with a `ModuleNotFoundError` (the cell
is tagged `raises-exception`).

In [ ]:
# Illustrative pseudocode—will not run as-is; MCP client setup and server details are omitted on purpose.
# It assumes a fictional 'mcp_client' library that handles the standardized
# communication with running MCP servers.

from mcp_client import MCPClient
import asyncio

async def run_mcp_agent():
    # 1. Initialize the client.
    # In a real scenario, this client would be configured to know the
    # addresses of available MCP servers (e.g., from a config file).
    client = MCPClient()

    # 2. The Agent's Brain (The LLM) is given a list of available tools.
    # The MCP client first gets the descriptions of all connected tools
    # (e.g., 'internal_charting_tool', 'community_search_tool')
    # and provides this "menu" to the LLM.
    available_tools = await client.list_tools()
    print(f"Agent now has access to tools: {[tool['name'] for tool in available_tools]}")

    # 3. The LLM processes the prompt and decides which tools to use.
    # This is where the AI's reasoning happens. It decides the sequence of actions.
    user_prompt = "Generate a report on $TECHCO: pull the latest chart and summarize recent community sentiment."

    # The client sends the prompt and the tool menu to the LLM. The LLM then
    # instructs the client which tool to call with which arguments.
    # The client handles the standardized communication (e.g., JSON-RPC call)
    # to the respective MCP server.
    print(f"LLM is now processing the prompt to create a plan...")
    result = await client.process(user_prompt, tools=available_tools)

    print("\n--- Agent's Final Report ---")
    print(result)

# In a real application, you would run this in an async environment.
# asyncio.run(run_mcp_agent())
print("MCP agent workflow completed conceptually.")